# Libraries

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install azure-storage-blob

In [0]:
import os
import requests

from azure.storage.blob import ContainerClient

from src.config.settings import (
    STORAGE_ACCOUNT_NAME,
    CONTAINER_NAME,
    SECRET_SCOPE,
    SECRET_KEY,
    LOCAL_LANDING_PATH,
    AZURE_LANDING_PREFIX,
    SOURCE_BASE_URL
)

### Parâmetros

In [0]:
dbutils.widgets.text("source_start_year", "2023", "Ano inicial")
dbutils.widgets.text("source_start_month", "1", "Mês inicial")
dbutils.widgets.text("source_end_year", "2023", "Ano final")
dbutils.widgets.text("source_end_month", "5", "Mês final")
dbutils.widgets.multiselect(
    "taxi_types",
    "yellow",
    ["yellow", "green"],
    "Tipos de Táxi"
)

# Recuperando os parametros da widget

SOURCE_START_YEAR = int(dbutils.widgets.get("source_start_year"))
SOURCE_START_MONTH = int(dbutils.widgets.get("source_start_month"))
SOURCE_END_YEAR = int(dbutils.widgets.get("source_end_year"))
SOURCE_END_MONTH = int(dbutils.widgets.get("source_end_month"))
TAXI_TYPES = [
    taxi_type.strip()
    for taxi_type in dbutils.widgets.get("taxi_types").split(",")
    if taxi_type.strip() 
]

# Functions

In [0]:
def generate_months(
    start_year: int,
    start_month: int,
    end_year: int,
    end_month: int
) -> list[tuple[int, int]]:
    """
    Gera uma lista de ano e mês entre o período inicial e final setados no arquivo de config.
    """

    months = []

    current_year = start_year
    current_month = start_month

    while (
        current_year < end_year
        or current_year == end_year and current_month <= end_month
    ):
        months.append((current_year, current_month))

        current_month += 1

        if current_month > 12:
            current_month = 1
            current_year += 1

    return months


def generate_source_files() -> list[str]:
    """
    Gera automaticamente os nomes dos arquivos da NYC TLC
    com base nos tipos de táxi e no período configurado.
    """

    source_files = []

    months = generate_months(
        SOURCE_START_YEAR,
        SOURCE_START_MONTH,
        SOURCE_END_YEAR,
        SOURCE_END_MONTH
    )

    for taxi_type in TAXI_TYPES:
        for year, month in months:
            file_name = f"{taxi_type}_tripdata_{year}-{month:02d}.parquet"
            source_files.append(file_name)

    return source_files

def get_sas_token() -> str:
    """
    Recupera o SAS Token salvo no Databricks Secret Scope.
    """

    sas_token = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key=SECRET_KEY
    )

    if sas_token.startswith("?"):
        sas_token = sas_token[1:]

    return sas_token


def get_container_client() -> ContainerClient:
    """
    Cria o client do Azure Blob Storage usando SAS Token.
    """

    account_url = f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net"

    return ContainerClient(
        account_url=account_url,
        container_name=CONTAINER_NAME,
        credential=get_sas_token()
    )


def create_local_landing_directory() -> None:
    """
    Cria a pasta local usada como Landing temporária.

    """
    os.makedirs(LOCAL_LANDING_PATH, exist_ok=True)


def blob_exists(container_client: ContainerClient, blob_name: str) -> bool:
    """
    Verifica se o arquivo já existe no Azure Blob Storage.
    """

    blob_client = container_client.get_blob_client(blob_name)

    return blob_client.exists()


def upload_file_to_azure(
    container_client: ContainerClient,
    local_file_path: str,
    blob_name: str
) -> None:
    """
    Faz upload de um arquivo local para o Azure Blob Storage.
    """

    blob_client = container_client.get_blob_client(blob_name)

    with open(local_file_path, "rb") as file:
        blob_client.upload_blob(file, overwrite=True)

    print(f"Arquivo enviado para Azure Landing: {blob_name}")


def download_file_to_local(url: str, local_file_path: str) -> None:
    """
    Faz download do arquivo da NYC TLC para a Landing local.
    Função criada apenas devido ao processamento limitado do Community Edition, 
    """

    print(f"Iniciando download: {url}")

    response = requests.get(url, stream=True, timeout=300)

    if response.status_code != 200:
        raise Exception(
            f"Erro ao baixar arquivo: {url}. "
            f"Status code: {response.status_code}"
        )

    with open(local_file_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    file_size_mb = round(os.path.getsize(local_file_path) / 1024 / 1024, 2)

    if file_size_mb <= 0:
        raise ValueError(f"Arquivo baixado está vazio: {local_file_path}")

    print(f"Arquivo baixado localmente: {local_file_path} | {file_size_mb} MB")


def ingestion_to_landing() -> None:
    """
    Baixa automaticamente os arquivos conforme tipos 
    e período definidos.

    Os arquivos são salvos localmente e enviados para a Landing Zone
    no Azure Blob Storage.

    O envio para a Azure aqui é apenas demonstrativo, uma vez que o
    Databricks free Edition nao le os arquivos do ADLS como fonte.
    """

    create_local_landing_directory()

    container_client = get_container_client()

    source_files = generate_source_files()

    for file_name in source_files:
        url = f"{SOURCE_BASE_URL}/{file_name}"

        local_file_path = os.path.join(LOCAL_LANDING_PATH, file_name)
        blob_name = f"{AZURE_LANDING_PREFIX}/{file_name}"

        if os.path.exists(local_file_path) and os.path.getsize(local_file_path) > 0:
            print(f"Arquivo já existe localmente. Pulando download: {local_file_path}")
        else:
            download_file_to_local(url, local_file_path)

        if blob_exists(container_client, blob_name):
            print(f"Arquivo já existe no Azure Landing. Pulando upload: {blob_name}")
        else:
            upload_file_to_azure(container_client, local_file_path, blob_name)

    print("Ingestão para Landing Zone concluída com sucesso.")

In [0]:
ingestion_to_landing()

In [0]:
dbutils.notebook.exit("OK")